# 01 — Data overview

Profiles the supplied synthetic source bundle. This notebook does not transform it into the canonical schema.

In [8]:
# Setup: Kaggle-compatible, deterministic, and CPU-safe.
from pathlib import Path
import logging, random
import numpy as np
import pandas as pd
random.seed(42); np.random.seed(42)
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
try:
    import torch
    logging.info('CUDA available: %s', torch.cuda.is_available())
except ImportError:
    logging.info('PyTorch unavailable; EDA uses CPU only.')
roots = [Path('/kaggle/input'), Path('data/original')]
candidates = [p for root in roots if root.exists() for p in root.rglob('transactions.csv')]
if not candidates:
    raise FileNotFoundError('Upload the source bundle to Kaggle Input or place it under data/original.')
source_dir = candidates[0].parent
transactions = pd.read_csv(source_dir / 'transactions.csv')
accounts = pd.read_csv(source_dir / 'accounts.csv')
features = pd.read_csv(source_dir / 'ml_features.csv')


INFO: CUDA available: False


In [9]:
display(pd.DataFrame({'file': ['transactions.csv', 'accounts.csv', 'ml_features.csv'], 'rows': [len(transactions), len(accounts), len(features)], 'columns': [transactions.shape[1], accounts.shape[1], features.shape[1]]}))
display(transactions.dtypes.rename('dtype').to_frame())
display(transactions.head())
print('Duplicate transaction rows:', transactions.duplicated().sum())
print('Missing values by field:')
display(transactions.isna().sum().loc[lambda s: s.gt(0)].rename('missing').to_frame())


,file,rows,columns
0,transactions.csv,100222,55
1,accounts.csv,65339,13
2,ml_features.csv,100222,35


,dtype
row_index,int64
Date,object
Time,object
Sender_account,int64
Receiver_account,int64
Amount,float64
Payment_currency,object
Received_currency,object
Sender_bank_location,object
Receiver_bank_location,object


,row_index,Date,Time,Sender_account,Receiver_account,Amount,Payment_currency,Received_currency,Sender_bank_location,Receiver_bank_location,...,tx_count_10,tx_count_30,amount_zscore,transmode_A,transmode_B,transmode_E,transmode_F,transmode_J,transmode_P,transmode_Z
0,0,2022-10-07,10:35:19,8724731955,2769355426,1459.15,UK pounds,UK pounds,UK,UK,...,1.0,1.0,-0.293518,1,0,0,0,0,0,0
1,1,2022-10-07,10:35:20,1491989064,8401255335,6019.64,UK pounds,Dirham,UK,UAE,...,1.0,1.0,-0.106146,0,0,0,1,0,0,0
2,2,2022-10-07,10:35:20,287305149,4404767002,14328.44,UK pounds,UK pounds,UK,UK,...,1.0,1.0,0.235230,0,0,1,0,0,0,0
3,3,2022-10-07,10:35:21,5376652437,9600420220,11895.00,UK pounds,UK pounds,UK,UK,...,1.0,1.0,0.135250,0,0,0,0,0,1,0
4,4,2022-10-07,10:35:21,9614186178,3803336972,115.25,UK pounds,UK pounds,UK,UK,...,1.0,1.0,-0.348734,1,0,0,0,0,0,0


Duplicate transaction rows: 0
Missing values by field:


,missing


## Conclusion
Use the source bundle for profiling only. Record every source-to-canonical mapping and recompute derived features during Phase 2.